# Nirṇaya — Colab Quickstart

Nirṇaya ("निर्णय", *decision*) is an India-first **System-One decision model**. Given a `state` (text — often Indic, code-mixed, or Hinglish) plus one or more typed questions, Nirṇaya returns calibrated probabilities in a single forward pass, with no text generation.

This notebook is **self-contained** — run every cell in order and you have a working Nirṇaya endpoint in ~5 minutes.

## Runtime requirement

Nirṇaya v1.0 needs **bf16** support, which means a GPU with compute capability ≥ 8.0 (Ampere or newer). On Colab that means **A100** or **L4** (both under Colab Pro). The default free-tier **T4 will NOT work** — Cell 1 below will tell you explicitly.

Switch runtime: *Runtime → Change runtime type → GPU → A100 (or L4) → Save*.

- Model card: https://huggingface.co/cmul8-hf/nirnaya
- Source repo: https://github.com/cmul8/nirnaya
- Licence: CC BY-NC 4.0 (research, evaluation, personal use). Commercial use — contact basab@cmul8.work.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmul8/nirnaya/blob/v0.1.0/examples/nirnaya_quickstart.ipynb)

## 1. Preflight — is this GPU compatible?

In [ ]:
import sys
import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print()

if not torch.cuda.is_available():
    print("No CUDA GPU available.")
    print("Fix: Runtime > Change runtime type > GPU > A100 (or L4).")
else:
    props = torch.cuda.get_device_properties(0)
    cc = props.major * 10 + props.minor
    print(f"GPU               : {props.name}")
    print(f"Compute capability: {props.major}.{props.minor}")
    print(f"VRAM              : {props.total_memory / 1e9:.1f} GB")
    print()
    if cc < 80:
        print("WARNING: this GPU does NOT support bf16 natively.")
        print("Nirṇaya v1.0 needs Ampere or newer (compute capability >= 8.0).")
        print("Fix: Runtime > Change runtime type > GPU > A100 (or L4).")
    else:
        print("bf16 supported. You are ready to install Nirṇaya.")

## 2. Install Nirṇaya

Colab already ships with a matching PyTorch build, so we skip the `torch` install and only install Nirṇaya and its light-weight deps (`transformers`, `peft`, `accelerate`, `huggingface-hub`, `sentencepiece`, `safetensors`, `tqdm`).

This cell may take ~1 minute.

In [ ]:
!pip install --quiet git+https://github.com/cmul8/nirnaya.git@v0.1.0
print("Installed.")

## 3. Load the model

The first call to `from_pretrained` downloads:

- **Qwen3-4B backbone** — ~8 GB (one-time)
- **Nirṇaya adapter + heads** — ~200 MB

Expect **2–5 minutes** on Colab's network the first time you run this. Colab caches downloads for the session; re-running is near-instant.

In [ ]:
from nirnaya import NirnayaModel

model = NirnayaModel.from_pretrained("cmul8-hf/nirnaya")
print("Loaded on:", next(model.parameters()).device)

## 4. A first decision — Hinglish support ticket

Three typed questions about the same state, all answered in **one forward pass**:

- `choice`: which department?
- `score` (1–5): how urgent?
- `noul`: escalate to a human?

In [ ]:
result = model.decide(
    state="बिजली बिल का पेमेंट अभी तक नहीं हुआ, urgent hai please. Kal se light nahi hai, entire building affected.",
    questions=[
        {"type": "choice", "options": ["billing", "technical", "sales"]},
        {"type": "score",  "k": 5, "instructions": "How urgent is this?"},
        {"type": "noul",   "instructions": "Should this be escalated?"},
    ],
)

for i, d in enumerate(result):
    probs = [round(p, 3) for p in d.probs]
    line = f"Q{i} [{d.type:6s}] probs={probs}  answer={d.answer}  confidence={d.confidence:.3f}"
    if d.expected_level is not None:
        line += f"  expected_level={d.expected_level:.2f}"
    if d.p_true is not None:
        line += f"  p_true={d.p_true:.3f}"
    print(line)

## 5. More languages, more shapes

The same API, on Tamil, Hindi, and English states.

In [ ]:
examples = [
    {
        "label": "Tamil — e-commerce return",
        "state": "நான் நேற்று வாங்கிய ஃபோன் screen crack ஆகி இருக்கு, refund வேணும். Order was delivered yesterday only.",
        "questions": [
            {"type": "choice", "options": ["order_status", "return_refund", "product_info", "delivery"]},
            {"type": "score", "k": 5, "instructions": "How urgent is this?"},
        ],
    },
    {
        "label": "Hindi (Devanagari) — UPI dispute",
        "state": "मेरे अकाउंट से पैसे कट गए लेकिन transaction fail हो गया। तीन घंटे हो गए, अभी तक refund नहीं मिला। बहुत परेशान हूँ।",
        "questions": [
            {"type": "score", "k": 5, "instructions": "How frustrated is the customer?"},
            {"type": "noul", "instructions": "Does this need a human agent?"},
        ],
    },
    {
        "label": "English — enterprise incident",
        "state": "Our ERP integration API has been returning 500 errors since last night. Finance team can't close the month-end books. Multiple downstream reports affected.",
        "questions": [
            {"type": "choice", "options": ["support", "sales", "billing", "engineering"]},
            {"type": "noul", "instructions": "Is this a P0 incident?"},
        ],
    },
]

for ex in examples:
    print(f"--- {ex['label']} ---")
    print(f"State: {ex['state'][:90]}{'...' if len(ex['state']) > 90 else ''}")
    result = model.decide(state=ex["state"], questions=ex["questions"])
    for i, d in enumerate(result):
        line = f"  Q{i} [{d.type:6s}] answer={str(d.answer):5s}  confidence={d.confidence:.3f}"
        if d.expected_level is not None:
            line += f"  expected_level={d.expected_level:.2f}"
        if d.p_true is not None:
            line += f"  p_true={d.p_true:.3f}"
        print(line)
    print()

## 6. Latency benchmark

How fast is a decision? And what does batching buy you? All questions in a single `decide()` call share **one** backbone forward pass — expect 1.5–3× vs. calling `decide()` per question.

In [ ]:
import statistics
import time

STATE = "बिजली बिल का पेमेंट अभी तक नहीं हुआ, urgent hai please. Kal se light nahi hai."
QUESTIONS = [
    {"type": "choice", "options": ["billing", "technical", "sales"]},
    {"type": "score",  "k": 5, "instructions": "How urgent is this?"},
    {"type": "noul",   "instructions": "Should this be escalated?"},
]


def time_fn(fn, n_warmup=3, n_timed=30):
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize()
    samples_ms = []
    for _ in range(n_timed):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        torch.cuda.synchronize()
        samples_ms.append((time.perf_counter() - t0) * 1000.0)
    return samples_ms


batched = time_fn(lambda: model.decide(state=STATE, questions=QUESTIONS))


def loop_call():
    for q in QUESTIONS:
        model.decide(state=STATE, questions=[q])


loop = time_fn(loop_call)

print("3-question decide (n=30, warmup=3):")
print(f"  batched  p50={statistics.median(batched):6.1f} ms  mean={statistics.mean(batched):6.1f} ms")
print(f"  loop     p50={statistics.median(loop):6.1f} ms  mean={statistics.mean(loop):6.1f} ms")
print()
print(f"Speedup (loop / batched, p50): {statistics.median(loop) / statistics.median(batched):.2f}x")

## 7. Your turn

Edit the cell below and rerun. Any mix of `choice`, `score` (with `k=` or `options=`) and `noul` works — up to 64 options per choice/score question.

In [ ]:
your_state = "YOUR TEXT HERE — Indic, Hinglish, or English are all fine."
your_questions = [
    {"type": "choice", "options": ["option_a", "option_b", "option_c"]},
    # {"type": "score", "k": 5, "instructions": "How urgent is this?"},
    # {"type": "noul",  "instructions": "Should this be escalated?"},
]

result = model.decide(state=your_state, questions=your_questions)
for d in result:
    print(d)

## What Nirṇaya *doesn't* do (v1.0)

- No text generation — decisions only.
- No CPU / MPS — CUDA-only.
- No int4 / int8 quantised inference (planned).
- No CLI or HTTP server. The Python API above is the whole surface.

## Next steps

- **Full model card:** https://huggingface.co/cmul8-hf/nirnaya
- **Issues, contributions:** https://github.com/cmul8/nirnaya
- **Commercial licence:** basab@cmul8.work